# 20 — FastText

FastText extends Word2Vec with **subword information**: each word's vector is built from the vectors of its character n-grams (e.g. "python" → "<py", "pyt", "yth", ..., "on>"). Unknown words are never truly "out of vocabulary" — their vectors are synthesized from their parts.

**Why it matters for resumes / ATS:** resumes are full of typos ("Tensorflo", "Pytorch", "progamming") and rare technical terms. Word2Vec throws a `KeyError` on those; FastText infers a vector, matches them to the right skill, and keeps the matcher alive.

**Goal:** Handle rare words and typos using subword information.

This chapter trains FastText on a tiny corpus, shows vectors being recovered for misspelled words that Word2Vec would reject, measures typo similarity with cosine, and digs into the character n-grams that make it all work. By the end you can explain why a typo like "pyton" still lands near "python". 

## 1. Training FastText with Gensim

The gensim API is nearly identical to Word2Vec: `FastText(sentences, vector_size=50, window=3, min_count=1, epochs=100)`. The difference is internal — alongside whole-word vectors it learns vectors for character n-grams, so every word is represented by a sum of its parts.

**What the code does:** trains on four sentences and reports vocabulary size and vector shape.
- Vocabulary: `15` tokens; `ft.wv['python']` has shape `(50,)`, the same layout as Word2Vec — the upgrade is invisible until you query an unknown word (next section).

**Try it:** the training API is identical to Ch. 19 — the subword machinery is what changed, not the interface.

In [ ]:
from gensim.models import FastText
sentences = [
    ["python", "deep", "learning", "framework"],
    ["nlp", "with", "python", "is", "powerful"],
    ["data", "science", "uses", "machine", "learning"],
    ["python", "programming", "language", "for", "data"],
]
ft = FastText(sentences, vector_size=50, window=3, min_count=1, epochs=100)
print(f"Vocabulary: {len(ft.wv)}")
print(f"Vector for 'python': shape {ft.wv['python'].shape}")

## 2. FastText Handles Out-of-Vocabulary Words

The payoff: query a word the trainer never saw and FastText still returns a vector, composed from its character n-grams. `pythoning`, `pyton`, `tensorflo`, `lernin`, `progamming` are all recoverable — and the model's nearest neighbors show it understands them.

**What the code does:** probes five misspellings and prints whether a vector was found and the top-2 neighbors.
- All five → `VECTOR FOUND (via subwords)`. This run: `'pythoning'` → `python (0.749)`, `'progamming'` → `programming (0.478)` — the typo maps straight back to the correct skill.

**Try it:** Word2Vec (Ch. 19) would raise `KeyError` on every one of these. That single difference — graceful degradation on messy input — is why FastText is the resume-parsing favorite.

In [ ]:
# Word2Vec would fail here — FastText uses subword n-grams
oov_words = ["pythoning", "pyton", "tensorflo", "lernin", "progamming"]
for w in oov_words:
    try:
        vec = ft.wv[w]  # FastText can infer from subwords
        print(f"  '{w}' -> VECTOR FOUND (via subwords)")
    except KeyError:
        print(f"  '{w}' -> NOT FOUND")

# Compare with similar known word
for w in oov_words:
    if w in ft.wv:
        similar = ft.wv.most_similar(w, topn=2)
        print(f"  '{w}' similar to: {similar}")

## 3. FastText vs Word2Vec on Typos

Quantifying the advantage: cosine similarity between a correct word and its misspelling. High scores mean the typo is "close enough" to the real term to match safely.

**What the code does:** scores three (correct, typo) pairs and contrasts with Word2Vec behavior.
- This run: `sim('python', 'pyton') = 0.233`, `sim('learning', 'lernin') = 0.114`, `sim('tensorflow', 'tensorflo') = 0.837` — the more shared character n-grams, the higher the score.
- Word2Vec: `KeyError` on all three — the contrast line the notebook prints.

**Try it:** `tensorflow`/`tensorflo` share almost every n-gram, hence 0.837; `lernin` loses more subwords, hence 0.114. That gradient is exactly what a typo-tolerant skill matcher should exploit.

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Correct vs misspelled
pairs = [("python", "pyton"), ("learning", "lernin"), ("tensorflow", "tensorflo")]
for correct, typo in pairs:
    if correct in ft.wv and typo in ft.wv:
        sim = cosine_similarity([ft.wv[correct]], [ft.wv[typo]])[0][0]
        print(f"  sim('{correct}', '{typo}') = {sim:.3f}")

# Compare with Word2Vec
print("\nWord2Vec would throw KeyError on typos.")
print("FastText gracefully handles misspellings via subword n-grams.")

## 4. Subword Details

Character n-grams with special boundary markers (`<`, `>`) are what make all of this work. "python" at n=3..6 produces `<py`, `pyt`, `yth`, `tho`, `hon`, `on>`, `<pyt`, `pyth`, ..., `ython>` — and "pyton" shares most of them.

**What the code does:** calls `compute_ngrams(word, 3, 6)` and prints the n-gram list.
- `compute_ngrams('python', 3, 6)` returns `['<py', 'pyt', 'yth', 'tho', 'hon', 'on>', '<pyt', 'pyth', 'ytho', 'thon', 'hon>', ...]` — 16 subword units per word.

**Try it (known issue):** on gensim ≥ 4.0 the guard `hasattr(ft.wv, 'ngrams')` is False — the attribute was removed — so the in-notebook listing is skipped and only the header and conclusion print. Call `compute_ngrams` directly (as shown) to see the list.

In [ ]:
# FastText stores character n-grams
word = "python"
print(f"Subword n-grams for '{word}' (n=3):")
if hasattr(ft.wv, 'ngrams'):
    from gensim.models.fasttext import compute_ngrams
    ngrams = compute_ngrams(word, 3, 6)
    print(f"  {ngrams}")

# This means "pyton" shares many n-grams with "python" -> similar vectors
print("\nThis is why FastText handles typos — shared character sequences!")

## Summary: Use FastText for resume parsing — resumes often have typos ('Tensorflo', 'Pytorch').

**Subword embeddings buy typo tolerance at the price of a larger, slower model — the right trade for messy resume text.**

FastText keeps Word2Vec's semantic vectors (Ch. 19) and adds character-level robustness, so "progamming" still matches "programming". It is the last self-trained embedding in the series: Ch. 21 (GloVe) shows how to leverage *pretrained* embeddings at scale, and Ch. 22 moves to contextual sentence embeddings entirely.